# 🔥 FireRed-Image-Edit — Free (T4)
T4 Optimized · Gradio UI · Google Drive I/O · Bulk · Multi-Prompt

> ⚠️ `Runtime → Change runtime type → GPU → T4`

In [ ]:
import torch,gc,os
!nvidia-smi
print(f'\nPyTorch:{torch.__version__}|CUDA:{torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU:{torch.cuda.get_device_name(0)}|VRAM:{torch.cuda.get_device_properties(0).total_mem/1024**3:.1f}GB')
from google.colab import drive
drive.mount('/content/drive')
def clear_mem():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
def vram():
    if torch.cuda.is_available(): return f'{torch.cuda.memory_allocated()/1024**3:.1f}/{torch.cuda.get_device_properties(0).total_mem/1024**3:.1f}GB'
    return 'N/A'

In [ ]:
!pip install -q diffusers accelerate Pillow tqdm cache_dit optimum-quanto google-generativeai gradio
if not os.path.exists('/content/FireRed-Image-Edit'):
    !git clone https://github.com/FireRedTeam/FireRed-Image-Edit.git /content/FireRed-Image-Edit
os.chdir('/content/FireRed-Image-Edit')
DOUT='/content/drive/MyDrive/FireRed-Output'; os.makedirs(DOUT,exist_ok=True)
print(f'✅ Output:{DOUT}')

In [ ]:
from diffusers import QwenImageEditPlusPipeline
MODEL_ID='FireRedTeam/FireRed-Image-Edit-1.0'
clear_mem()
print(f'⏳ Loading {MODEL_ID} T4 optimized...')
try: pipe=QwenImageEditPlusPipeline.from_pretrained(MODEL_ID,torch_dtype=torch.float16); print('✅ float16')
except: pipe=QwenImageEditPlusPipeline.from_pretrained(MODEL_ID,torch_dtype=torch.bfloat16); print('✅ bfloat16')
pipe.enable_model_cpu_offload(); pipe.enable_attention_slicing('max')
pipe.set_progress_bar_config(disable=None)
print(f'🎉 Ready! VRAM:{vram()}')

In [ ]:
import gradio as gr
import glob,time
from PIL import Image
from datetime import datetime

DR='/content/drive/MyDrive'

def smart_resize(img,mx=768):
    w,h=img.size
    if max(w,h)>mx: r=mx/max(w,h); w,h=(int(w*r)//16)*16,(int(h*r)//16)*16; img=img.resize((w,h),Image.LANCZOS)
    else:
        nw,nh=(w//16)*16,(h//16)*16
        if nw!=w or nh!=h: img=img.resize((nw,nh),Image.LANCZOS)
    return img

def browse_drive(fp):
    if not fp: fp=DR
    if not os.path.isabs(fp): fp=os.path.join(DR,fp)
    if not os.path.exists(fp): return [],f'❌ {fp}'
    exts=('*.png','*.jpg','*.jpeg','*.webp','*.bmp')
    files=[]
    for e in exts: files.extend(glob.glob(os.path.join(fp,e))); files.extend(glob.glob(os.path.join(fp,e.upper())))
    files=sorted(set(files))[:100]
    subs=[d for d in sorted(os.listdir(fp)) if os.path.isdir(os.path.join(fp,d)) and not d.startswith('.')]
    return files,f'📂 {fp}\n🖼️ {len(files)} gambar' + (f'\n📁 {', '.join(subs[:20])}' if subs else '')

def load_from_drive(fp):
    if not fp: return None,'❌'
    if not os.path.isabs(fp): fp=os.path.join(DR,fp)
    if not os.path.exists(fp): return None,f'❌ {fp}'
    try: img=Image.open(fp).convert('RGB'); return img,f'✅ {os.path.basename(fp)} ({img.size[0]}x{img.size[1]})'
    except Exception as e: return None,f'❌ {e}'

def infer(images,prompt,seed,cfg,steps,mx):
    resized=[smart_resize(i,int(mx)) for i in images]
    with torch.inference_mode():
        return pipe(image=resized,prompt=prompt,generator=torch.Generator(device='cpu').manual_seed(int(seed)),
            true_cfg_scale=float(cfg),negative_prompt=' ',num_inference_steps=int(steps),num_images_per_prompt=1).images[0]

def edit_single(inp,prompt,seed,cfg,steps,mx,save):
    if inp is None: return None,'❌ Upload!'
    if not prompt.strip(): return None,'❌ Prompt!'
    t=time.time(); clear_mem()
    try:
        out=infer([Image.fromarray(inp).convert('RGB')],prompt,seed,cfg,steps,mx)
        s=f'✅ ({time.time()-t:.1f}s)\n'
        if save: p=os.path.join(DOUT,f'firered_{datetime.now().strftime("%Y%m%d_%H%M%S")}.png'); out.save(p); s+=f'💾 {p}\n'
        s+=f'VRAM:{vram()}'; return out,s
    except RuntimeError as e:
        if 'out of memory' in str(e).lower(): clear_mem(); return None,'❌ OOM! Kurangi Res/Steps.'
        return None,f'❌ {e}'

def edit_multi(i1,i2,i3,prompt,seed,cfg,steps,mx,save):
    imgs=[Image.fromarray(x).convert('RGB') for x in [i1,i2,i3] if x is not None]
    if not imgs: return None,'❌'
    if not prompt.strip(): return None,'❌ Prompt!'
    t=time.time(); clear_mem()
    try:
        out=infer(imgs,prompt,seed,cfg,steps,mx)
        s=f'✅ ({time.time()-t:.1f}s) {len(imgs)} imgs\n'
        if save: p=os.path.join(DOUT,f'firered_multi_{datetime.now().strftime("%Y%m%d_%H%M%S")}.png'); out.save(p); s+=f'💾 {p}\n'
        return out,s
    except RuntimeError as e:
        if 'out of memory' in str(e).lower(): clear_mem(); return None,'❌ OOM!'
        return None,f'❌ {e}'

def bulk_edit(inf,outf,prompt,seed,cfg,steps,mx,progress=gr.Progress()):
    if not prompt.strip() or not inf or not outf: return [],'❌ Input/Output/Prompt!'
    inp=inf if os.path.isabs(inf) else os.path.join(DR,inf)
    outp=outf if os.path.isabs(outf) else os.path.join(DR,outf)
    if not os.path.exists(inp): return [],f'❌ {inp}'
    os.makedirs(outp,exist_ok=True)
    exts=('*.png','*.jpg','*.jpeg','*.webp','*.bmp')
    files=[]
    for e in exts: files.extend(glob.glob(os.path.join(inp,e))); files.extend(glob.glob(os.path.join(inp,e.upper())))
    files=sorted(set(files))
    if not files: return [],'❌ No images'
    res=[]; log=f'📦 {len(files)} gambar\n🎨 {prompt[:60]}\n\n'; T=time.time(); ok=0
    for i,fp in enumerate(progress.tqdm(files,desc='Bulk')):
        fn=os.path.basename(fp); log+=f'[{i+1}/{len(files)}] {fn}... '
        try:
            img=Image.open(fp).convert('RGB'); clear_mem(); t=time.time()
            out=infer([img],prompt,seed,cfg,steps,mx)
            out.save(os.path.join(outp,fn)); res.append(os.path.join(outp,fn)); ok+=1
            log+=f'✅ ({time.time()-t:.1f}s)\n'
        except: clear_mem(); log+='❌\n'
    log+=f'\n🎉 {ok}/{len(files)} | ⏱️ {time.time()-T:.0f}s | 💾 {outp}'
    return res,log

def multi_prompt_single(inp,prompts_text,seed,cfg,steps,mx,outf,progress=gr.Progress()):
    if inp is None: return [],'❌ Upload!'
    prompts=[p.strip() for p in prompts_text.strip().split('\n') if p.strip()]
    if not prompts: return [],'❌ Prompt!'
    outp=outf if os.path.isabs(outf) else os.path.join(DR,outf)
    os.makedirs(outp,exist_ok=True)
    img=Image.fromarray(inp).convert('RGB')
    res=[]; log=f'🎨 {len(prompts)} prompt × 1 gambar\n\n'; T=time.time(); ok=0
    for i,p in enumerate(progress.tqdm(prompts,desc='Prompts')):
        log+=f'[{i+1}/{len(prompts)}] "{p[:50]}"... '
        try:
            clear_mem(); t=time.time(); out=infer([img],p,seed,cfg,steps,mx)
            fn=f'prompt_{i+1}.png'; out.save(os.path.join(outp,fn)); res.append(os.path.join(outp,fn)); ok+=1
            log+=f'✅ ({time.time()-t:.1f}s)\n'
        except: clear_mem(); log+='❌\n'
    log+=f'\n🎉 {ok}/{len(prompts)} | ⏱️ {time.time()-T:.0f}s | 💾 {outp}'
    return res,log

def multi_prompt_bulk(inf,prompts_text,seed,cfg,steps,mx,outf,progress=gr.Progress()):
    if not inf: return [],'❌ Folder!'
    prompts=[p.strip() for p in prompts_text.strip().split('\n') if p.strip()]
    if not prompts: return [],'❌ Prompt!'
    inp=inf if os.path.isabs(inf) else os.path.join(DR,inf)
    outp=outf if os.path.isabs(outf) else os.path.join(DR,outf)
    if not os.path.exists(inp): return [],f'❌ {inp}'
    exts=('*.png','*.jpg','*.jpeg','*.webp','*.bmp')
    files=[]
    for e in exts: files.extend(glob.glob(os.path.join(inp,e))); files.extend(glob.glob(os.path.join(inp,e.upper())))
    files=sorted(set(files))
    if not files: return [],'❌ No images'
    total=len(files)*len(prompts)
    log=f'📦 {len(files)} gambar × {len(prompts)} prompt = {total} output\n\n'
    res=[]; T=time.time(); ok=0; n=0
    for pi,p in enumerate(prompts):
        pdir=os.path.join(outp,f'prompt_{pi+1}'); os.makedirs(pdir,exist_ok=True)
        log+=f'--- Prompt {pi+1}: "{p[:60]}" ---\n'
        for fi,fp in enumerate(progress.tqdm(files,desc=f'P{pi+1}')):
            fn=os.path.basename(fp); n+=1; log+=f'  [{n}/{total}] {fn}... '
            try:
                img=Image.open(fp).convert('RGB'); clear_mem(); t=time.time()
                out=infer([img],p,seed,cfg,steps,mx)
                out.save(os.path.join(pdir,fn)); res.append(os.path.join(pdir,fn)); ok+=1
                log+=f'✅ ({time.time()-t:.1f}s)\n'
            except: clear_mem(); log+='❌\n'
    log+=f'\n🎉 {ok}/{total} | ⏱️ {time.time()-T:.0f}s | 💾 {outp}'
    return res,log

# ============ UI ============
THEME=gr.themes.Soft(primary_hue='orange',secondary_hue='amber',neutral_hue='slate',font=[gr.themes.GoogleFont('Inter'),'sans-serif'])
CSS='.gradio-container{max-width:1200px!important}.mt{text-align:center}.mt h1{color:#ea580c;font-size:2em}.sb{font-family:monospace;font-size:.85em}'

with gr.Blocks(theme=THEME,css=CSS,title='FireRed Free') as demo:
    gr.HTML("<div class='mt'><h1>🔥 FireRed Image Edit — Free (T4)</h1><p style='color:#888'>T4 Optimized · Drive I/O · Bulk · Multi-Prompt</p></div>")
    with gr.Tabs():
        with gr.TabItem('🖼️ Single'):
            with gr.Row():
                with gr.Column():
                    si=gr.Image(label='Upload',type='numpy',height=380)
                    with gr.Accordion('📁 Drive',open=False):
                        sdp=gr.Textbox(label='Path'); slb=gr.Button('📂',size='sm'); sls=gr.Textbox(interactive=False,lines=1)
                    sp=gr.Textbox(label='🎨 Prompt',lines=3,value='Transform the object into a miniature held between thumb and forefinger.')
                    with gr.Row(): ss=gr.Number(label='Seed',value=49,precision=0); sc=gr.Slider(label='CFG',minimum=1,maximum=8,value=3.5,step=0.5)
                    with gr.Row(): sst=gr.Slider(label='Steps',minimum=8,maximum=50,value=28,step=1); smr=gr.Slider(label='MaxRes',minimum=384,maximum=1024,value=768,step=64)
                    ssv=gr.Checkbox(label='💾 Save',value=True); sb=gr.Button('🚀 Generate!',variant='primary',size='lg')
                with gr.Column(): so=gr.Image(label='Output',type='pil',height=380); sot=gr.Textbox(label='Status',interactive=False,lines=5,elem_classes='sb')
            slb.click(fn=load_from_drive,inputs=[sdp],outputs=[si,sls])
            sb.click(fn=edit_single,inputs=[si,sp,ss,sc,sst,smr,ssv],outputs=[so,sot])

        with gr.TabItem('🖼️ Multi-Img'):
            with gr.Row(): mi1=gr.Image(label='1',type='numpy',height=220); mi2=gr.Image(label='2',type='numpy',height=220); mi3=gr.Image(label='3',type='numpy',height=220)
            mp=gr.Textbox(label='🎨 Prompt',lines=3)
            with gr.Row(): ms=gr.Number(label='Seed',value=42,precision=0); mc=gr.Slider(label='CFG',minimum=1,maximum=8,value=3.5,step=0.5); mst2=gr.Slider(label='Steps',minimum=8,maximum=40,value=24,step=1); mmr=gr.Slider(label='MaxRes',minimum=384,maximum=768,value=512,step=64)
            msv=gr.Checkbox(label='💾 Save',value=True); mbn=gr.Button('🚀 Generate!',variant='primary',size='lg')
            with gr.Row(): mo=gr.Image(label='Output',type='pil',height=380); mot=gr.Textbox(interactive=False,lines=4)
            mbn.click(fn=edit_multi,inputs=[mi1,mi2,mi3,mp,ms,mc,mst2,mmr,msv],outputs=[mo,mot])

        with gr.TabItem('📦 Bulk'):
            gr.Markdown('### 1 Prompt → Banyak Gambar. Output nama file **sama**.')
            with gr.Row(): bf=gr.Textbox(label='📂 Input',scale=2); bfp=gr.Button('🔍',scale=1)
            bfi=gr.Textbox(interactive=False,lines=2); bfg=gr.Gallery(columns=6,height=150,object_fit='contain')
            bof=gr.Textbox(label='📂 Output',value='FireRed-Output/bulk')
            bp=gr.Textbox(label='🎨 Prompt',lines=2,value='Make it watercolor.')
            with gr.Row(): bsd=gr.Number(label='Seed',value=42,precision=0); bcf=gr.Slider(label='CFG',minimum=1,maximum=8,value=3.5,step=0.5); bstp=gr.Slider(label='Steps',minimum=8,maximum=40,value=24,step=1); bmr=gr.Slider(label='MaxRes',minimum=384,maximum=768,value=640,step=64)
            bb=gr.Button('📦 Start!',variant='primary',size='lg')
            bl=gr.Textbox(label='Log',interactive=False,lines=15,elem_classes='sb'); brs=gr.Gallery(columns=5,height=300,object_fit='contain')
            bfp.click(fn=browse_drive,inputs=[bf],outputs=[bfg,bfi])
            bb.click(fn=bulk_edit,inputs=[bf,bof,bp,bsd,bcf,bstp,bmr],outputs=[brs,bl])

        with gr.TabItem('🎨 Multi-Prompt'):
            gr.Markdown('### Banyak Prompt\nTulis 1 prompt per baris. Output di subfolder `prompt_1/`, `prompt_2/`...')
            with gr.Tabs():
                with gr.TabItem('1 Gambar × N Prompt'):
                    mps_img=gr.Image(label='Upload',type='numpy',height=300)
                    mps_p=gr.Textbox(label='Prompts (1/baris)',lines=6,placeholder='Make it watercolor\nMake it pencil sketch\nMake it pop art\nAdd snow')
                    with gr.Row(): mps_s=gr.Number(label='Seed',value=42,precision=0); mps_c=gr.Slider(label='CFG',minimum=1,maximum=8,value=3.5,step=0.5); mps_st=gr.Slider(label='Steps',minimum=8,maximum=40,value=24,step=1); mps_mr=gr.Slider(label='MaxRes',minimum=384,maximum=768,value=640,step=64)
                    mps_o=gr.Textbox(label='📂 Output',value='FireRed-Output/multi_prompt')
                    mps_b=gr.Button('🎨 Run!',variant='primary',size='lg')
                    mps_log=gr.Textbox(interactive=False,lines=10,elem_classes='sb'); mps_gal=gr.Gallery(columns=4,height=400,object_fit='contain')
                    mps_b.click(fn=multi_prompt_single,inputs=[mps_img,mps_p,mps_s,mps_c,mps_st,mps_mr,mps_o],outputs=[mps_gal,mps_log])

                with gr.TabItem('N Gambar × N Prompt'):
                    gr.Markdown('Output: `output/prompt_1/img.png`, `output/prompt_2/img.png`, ...')
                    with gr.Row(): mpb_f=gr.Textbox(label='📂 Input',scale=2); mpb_prev=gr.Button('🔍',scale=1)
                    mpb_info=gr.Textbox(interactive=False,lines=2); mpb_gin=gr.Gallery(columns=6,height=150,object_fit='contain')
                    mpb_p=gr.Textbox(label='Prompts (1/baris)',lines=6,placeholder='Make it watercolor\nMake it pencil sketch')
                    with gr.Row(): mpb_s=gr.Number(label='Seed',value=42,precision=0); mpb_c=gr.Slider(label='CFG',minimum=1,maximum=8,value=3.5,step=0.5); mpb_st=gr.Slider(label='Steps',minimum=8,maximum=40,value=24,step=1); mpb_mr=gr.Slider(label='MaxRes',minimum=384,maximum=768,value=640,step=64)
                    mpb_o=gr.Textbox(label='📂 Output',value='FireRed-Output/multi_prompt_bulk')
                    mpb_b=gr.Button('📦🎨 Run!',variant='primary',size='lg')
                    mpb_log=gr.Textbox(interactive=False,lines=15,elem_classes='sb'); mpb_gal=gr.Gallery(columns=5,height=400,object_fit='contain')
                    mpb_prev.click(fn=browse_drive,inputs=[mpb_f],outputs=[mpb_gin,mpb_info])
                    mpb_b.click(fn=multi_prompt_bulk,inputs=[mpb_f,mpb_p,mpb_s,mpb_c,mpb_st,mpb_mr,mpb_o],outputs=[mpb_gal,mpb_log])

        with gr.TabItem('📁 Drive'):
            with gr.Row(): gf=gr.Textbox(label='Folder',value='',scale=3); gb=gr.Button('🔍',variant='primary',scale=1)
            gi=gr.Textbox(interactive=False,lines=3); gg=gr.Gallery(columns=5,height=450,object_fit='contain')
            gb.click(fn=browse_drive,inputs=[gf],outputs=[gg,gi])

    gr.Markdown('---\n🔥 FireRed-Image-Edit-1.0')
demo.launch(share=True,debug=True)